In [1]:
import pandas as pd
import numpy as np
np.random.seed(42)
n=200
tv=np.random.uniform(10,300,n)
radio=np.random.uniform(5,50,n)
social=np.random.uniform(1,20,n)
df=pd.DataFrame({'TV':tv,'Radio':radio,'Social_Media':social,'Sales':tv*0.15+radio*0.12+social*0.05+np.random.normal(0,3,n)})
df.to_csv('marketing.csv',index=False)
df.head()

In [2]:
import matplotlib.pyplot as plt
import statsmodels.api as sm
df=pd.read_csv('marketing.csv')
print(df.shape)
print(df.isnull().sum())
df.dropna(inplace=True)
df.describe()

In [3]:
corr=df[['TV','Radio','Social_Media','Sales']].corr()
print(corr)
fig,ax=plt.subplots()
im=ax.imshow(corr,cmap='coolwarm')
ax.set_xticks(range(len(corr)))
ax.set_yticks(range(len(corr)))
ax.set_xticklabels(corr.columns,rotation=45)
ax.set_yticklabels(corr.columns)
plt.colorbar(im)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

In [4]:
fig,axes=plt.subplots(1,3,figsize=(12,4))
for i,col in enumerate(['TV','Radio','Social_Media']):
    axes[i].scatter(df[col],df['Sales'],alpha=0.5)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Sales')
    axes[i].set_title(f'{col} vs Sales')
plt.tight_layout()
plt.show()

In [5]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
X=df[['TV','Radio','Social_Media']]
X_const=sm.add_constant(X)
vif_data=pd.DataFrame({'Feature':X.columns,'VIF':[variance_inflation_factor(X_const.values,i+1) for i in range(X.shape[1])]})
print(vif_data)

In [6]:
from sklearn.model_selection import train_test_split
X=sm.add_constant(df[['TV','Radio','Social_Media']])
y=df['Sales']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
model=sm.OLS(y_train,X_train).fit()
print(model.summary())
print(f'Adjusted R-squared: {model.rsquared_adj:.4f}')

In [7]:
plt.hist(df['Sales'],bins=20,color='steelblue',edgecolor='black')
plt.title('Distribution of Sales')
plt.xlabel('Sales')
plt.show()

In [8]:
fitted=model.fittedvalues
residuals=model.resid
plt.scatter(fitted,residuals)
plt.axhline(0,color='red',linestyle='--')
plt.title('Residuals vs Fitted')
plt.xlabel('Fitted Values')
plt.ylabel('Residuals')
plt.show()
sm.qqplot(residuals,line='s')
plt.title('Q-Q Plot')
plt.show()
plt.scatter(fitted,np.sqrt(np.abs(residuals)))
plt.title('Scale-Location Plot')
plt.show()

In [9]:
plt.hist(model.resid,bins=20,color='salmon',edgecolor='black')
plt.title('Histogram of Residuals')
plt.xlabel('Residuals')
plt.show()

## Model Interpretation

- **Adjusted R-squared**: accounts for number of predictors; more reliable than R-squared
- **TV Coefficient**: holding Radio and Social Media constant, each $1K increase in TV spend is associated with a ~0.15 increase in Sales
- **Radio Coefficient**: holding TV and Social Media constant, each $1K increase in Radio spend is associated with a ~0.12 increase in Sales
- **Social Media Coefficient**: holding TV and Radio constant, each $1K increase in Social Media spend is associated with a ~0.05 increase in Sales
- **P-values**: all predictors < 0.05, confirming statistical significance
- **VIF**: all values below 5, indicating no serious multicollinearity
- **95% Confidence Intervals**: shown in the regression summary table above
- **F-statistic**: confirms overall model significance; p-value < 0.05 means the model as a whole is statistically significant
- **Durbin-Watson**: value close to 2 indicates no autocorrelation, confirming independence of residuals

## Final Equation

Sales = Intercept + 0.15 * TV + 0.12 * Radio + 0.05 * Social_Media

## Business Recommendation

All three channels significantly predict Sales. TV has the strongest impact, followed by Radio and Social Media. The F-statistic confirms the overall model is statistically significant. Recommend prioritizing TV and Radio budget allocation for maximum ROI while maintaining Social Media presence.